# Province Classifier (77 จังหวัด + เบตง)

เทรนตัวจำแนกจังหวัดจากภาพ lower crop ขนาด 128×32 เพื่อเอาไปใช้กับงาน inference แบบ real-time (เช่น RTSP).

- แนะนำเริ่ม: `tf_efficientnet_b0` (แม่นขึ้น)
- ถ้าต้องการเบา/เร็วมาก: `mobilenetv3_small_100`

Datasets (zip บน MyDrive): `lower_train.zip`, `lower_test.zip`, `lower_test_synthetic.zip`
ผลลัพธ์จะ save ลง `/content/drive/MyDrive/alpr_province_classifier/` เพื่อไม่ให้หายเมื่อ Colab reset.


In [28]:
# Cell 2: Dataset paths (Local laptop: use existing folders; Colab: optional zip/unzip)
import os, sys, zipfile, shutil
from pathlib import Path

def is_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

IS_COLAB = is_colab()
print('IS_COLAB:', IS_COLAB)

def find_repo_root(start: Path) -> Path:
    """Walk upwards to find a folder containing data/lower_train/labels.csv."""
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'data' / 'lower_train' / 'labels.csv').exists():
            return p
    return start  # fallback (will likely fail later with clear message)

# --- Local laptop mode (no unzip) ---
def resolve_local_dataset_dirs() -> tuple[str, str, str]:
    repo_root = find_repo_root(Path.cwd())
    data_root = repo_root / 'data'

    train_dir = data_root / 'lower_train'
    test_dir = data_root / 'lower_test'
    syn_dir = data_root / 'lower_test_synthetic'

    return str(train_dir), str(test_dir), str(syn_dir)

# --- Colab mode (optional unzip from Drive) ---
def unzip_to(zip_path: str, out_dir: str):
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f'Zip not found: {zip_path}')
    if os.path.exists(out_dir) and os.path.isdir(out_dir) and len(os.listdir(out_dir)) > 0:
        print(f'Skip (already exists): {out_dir}')
        return
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(out_dir)
    print(f'Unzipped: {zip_path} -> {out_dir}')

def resolve_dataset_dir(root_dir: str) -> str:
    # Supports both layouts:
    # 1) root_dir/labels.csv + root_dir/data/...
    # 2) root_dir/<subdir>/labels.csv + ...
    if os.path.exists(os.path.join(root_dir, 'labels.csv')):
        return root_dir
    for d in os.listdir(root_dir):
        sd = os.path.join(root_dir, d)
        if os.path.isdir(sd) and os.path.exists(os.path.join(sd, 'labels.csv')):
            return sd
    raise FileNotFoundError(f'Could not find labels.csv under: {root_dir}')

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    ZIP_TRAIN = '/content/drive/MyDrive/ALPRV2/lower_train.zip'
    ZIP_TEST = '/content/drive/MyDrive/ALPRV2/lower_test.zip'
    ZIP_TEST_SYN = '/content/drive/MyDrive/ALPRV2/lower_test_synthetic.zip'

    OUT_ROOT = '/content/datasets'
    RAW_TRAIN_DIR = os.path.join(OUT_ROOT, 'lower_train')
    RAW_TEST_DIR = os.path.join(OUT_ROOT, 'lower_test')
    RAW_TEST_SYN_DIR = os.path.join(OUT_ROOT, 'lower_test_synthetic')
    os.makedirs(OUT_ROOT, exist_ok=True)

    unzip_to(ZIP_TRAIN, RAW_TRAIN_DIR)
    unzip_to(ZIP_TEST, RAW_TEST_DIR)
    unzip_to(ZIP_TEST_SYN, RAW_TEST_SYN_DIR)

    TRAIN_DIR = resolve_dataset_dir(RAW_TRAIN_DIR)
    TEST_DIR = resolve_dataset_dir(RAW_TEST_DIR)
    TEST_SYN_DIR = resolve_dataset_dir(RAW_TEST_SYN_DIR)
else:
    TRAIN_DIR, TEST_DIR, TEST_SYN_DIR = resolve_local_dataset_dirs()

print('CWD          :', str(Path.cwd().resolve()))
print('TRAIN_DIR    :', str(Path(TRAIN_DIR).resolve()))
print('TEST_DIR     :', str(Path(TEST_DIR).resolve()))
print('TEST_SYN_DIR :', str(Path(TEST_SYN_DIR).resolve()))

missing = []
for p in [TRAIN_DIR, TEST_DIR, TEST_SYN_DIR]:
    if not os.path.exists(os.path.join(p, 'labels.csv')):
        missing.append(p)
if missing:
    raise FileNotFoundError('Missing labels.csv in: ' + ' | '.join(missing))

IS_COLAB: True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Skip (already exists): /content/datasets/lower_train
Skip (already exists): /content/datasets/lower_test
Skip (already exists): /content/datasets/lower_test_synthetic
CWD          : /content
TRAIN_DIR    : /content/datasets/lower_train/lower_train
TEST_DIR     : /content/datasets/lower_test/lower_test
TEST_SYN_DIR : /content/datasets/lower_test_synthetic/lower_test_synthetic


In [29]:
# Cell 3: Install dependencies (works on Colab + local Jupyter)
import sys, subprocess
pkgs = [
    'timm==0.9.16',
    'pandas==2.2.2',
    'scikit-learn==1.5.2',
    'tqdm==4.66.4',
    'pillow',
    'torchvision',
 ]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('Installed deps')

Installed deps


In [30]:
# Cell 4: Imports + seed
import os, json, random, time
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm.auto import tqdm

import torchvision.transforms as T

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
torch.backends.cudnn.benchmark = True
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

device: cuda


In [31]:
# Cell 5: Config
@dataclass
class CFG:
    # แนะนำเริ่มด้วย: 'tf_efficientnet_b0' (แม่นขึ้น)
    # ถ้าต้องการเบา/เร็วมาก: 'mobilenetv3_small_100'
    model_name: str = 'mobilenetv3_small_100'  # เปลี่ยนจาก tf_efficientnet_b0
    num_classes: int = 78

    img_h: int = 32
    img_w: int = 128

    # --- Debug run on laptop ---
    debug_run: bool = True        # True = รันสั้นๆ เพื่อเช็คว่าใช้งานได้
    debug_train_per_class: int = 150  # จำกัดจำนวนรูปต่อคลาสใน train เพื่อให้รันไว
    debug_eval_rows: int = 3000       # จำกัดจำนวนรูปตอน eval
    max_train_batches: int = 50       # จำกัดจำนวน batch ต่อ epoch (train)
    max_val_batches: int = 20         # จำกัดจำนวน batch ต่อ epoch (val)

    epochs: int = 20
    batch_size: int = 64
    lr: float = 3e-4
    weight_decay: float = 1e-4

    num_workers: int = 0
    val_ratio: float = 0.10

    amp: bool = True
    label_col_preference: tuple = ('province_description', 'label')
    # Remove "unknown" province entries in all splits (train/test/test_syn)
    excluded_labels: tuple = ('ไม่พบข้อมูล',)

    # Save: local -> ./artifacts ; colab -> MyDrive
    save_dir: str = './artifacts/alpr_province_classifier'

cfg = CFG()

# Auto adjust for Colab/GPU
if 'IS_COLAB' in globals() and IS_COLAB:
    cfg.save_dir = '/content/drive/MyDrive/ALPR_Project/alpr_province_classifier_final'
    cfg.batch_size = 256
    cfg.num_workers = 2
    cfg.persistent_workers = True # สั่งให้ workers รออยู่ตลอด ไม่ต้องสร้างใหม่ทุก epoch
    cfg.epochs = 20
    cfg.debug_run = False

os.makedirs(cfg.save_dir, exist_ok=True)
print('save_dir:', cfg.save_dir)
print('debug_run:', cfg.debug_run)
print('excluded_labels:', cfg.excluded_labels)

save_dir: /content/drive/MyDrive/ALPR_Project/alpr_province_classifier_final
debug_run: False
excluded_labels: ('ไม่พบข้อมูล',)


In [32]:
# Cell 6: Read labels + build label map
def read_labels(dataset_dir: str) -> pd.DataFrame:
    labels_path = os.path.join(dataset_dir, 'labels.csv')
    if not os.path.exists(labels_path):
        raise FileNotFoundError(f'labels.csv not found in {dataset_dir}')
    df = pd.read_csv(labels_path)
    if 'filename' not in df.columns:
        raise ValueError('labels.csv must contain a filename column')

    label_col = None
    for c in cfg.label_col_preference:
        if c in df.columns:
            label_col = c
            break
    if label_col is None:
        raise ValueError(f'labels.csv missing label column; tried {cfg.label_col_preference}')

    df = df[['filename', label_col]].copy()
    df.rename(columns={label_col: 'label'}, inplace=True)
    df['label'] = df['label'].astype(str).str.strip()

    # Remove unknown / missing province labels (e.g. "ไม่พบข้อมูล")
    if getattr(cfg, 'excluded_labels', None):
        before = len(df)
        df = df[~df['label'].isin(set(cfg.excluded_labels))].copy()
        removed = before - len(df)
        if removed > 0:
            print(f'[{Path(dataset_dir).name}] removed excluded_labels: {removed}')

    df['path'] = df['filename'].apply(lambda x: os.path.join(dataset_dir, 'data', str(x)))
    df = df[df['path'].apply(os.path.exists)].reset_index(drop=True)
    return df

train_df = read_labels(TRAIN_DIR)
print('train rows (raw, after exclude + exists):', len(train_df))

classes = sorted(train_df['label'].unique().tolist())
print('unique classes in train:', len(classes))

if len(classes) != cfg.num_classes:
    print(f'WARNING: expected {cfg.num_classes} classes but found {len(classes)} in train labels. Using detected count.')
    cfg.num_classes = len(classes)

label2idx = {c: i for i, c in enumerate(classes)}
idx2label = {i: c for c, i in label2idx.items()}

with open(os.path.join(cfg.save_dir, 'label_map.json'), 'w', encoding='utf-8') as f:
    json.dump({'label2idx': label2idx, 'idx2label': idx2label}, f, ensure_ascii=False, indent=2)

train_df['y'] = train_df['label'].map(label2idx).astype(int)

# Debug: จำกัดจำนวนรูปต่อคลาส เพื่อรันบน laptop ได้ไว
if cfg.debug_run:
    train_df = (
        train_df.groupby('y', group_keys=False)
                .apply(lambda g: g.sample(n=min(len(g), cfg.debug_train_per_class), random_state=42))
                .reset_index(drop=True)
    )
    print('train rows (debug sampled):', len(train_df))

train_df['y'].value_counts().head()

/tmp/ipython-input-3745098212.py:6: DtypeWarning: Columns (3,4,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(labels_path)


[lower_train] removed excluded_labels: 2
train rows (raw, after exclude + exists): 155999
unique classes in train: 78


,count
y,
31,2004
1,2000
8,2000
18,2000
15,2000


In [33]:
# Cell 7: Stratified split train/val (90/10)
sss = StratifiedShuffleSplit(n_splits=1, test_size=cfg.val_ratio, random_state=42)
train_idx, val_idx = next(sss.split(train_df['path'], train_df['y']))
tr_df = train_df.iloc[train_idx].reset_index(drop=True)
va_df = train_df.iloc[val_idx].reset_index(drop=True)

print('train split:', len(tr_df), 'val split:', len(va_df))
print('train classes:', tr_df['y'].nunique(), 'val classes:', va_df['y'].nunique())

train split: 140399 val split: 15600
train classes: 78 val classes: 78


In [34]:
# Cell 8: Dataset + DataLoader
train_tfm = T.Compose([
    T.Resize((cfg.img_h, cfg.img_w)),
    T.RandomApply([T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.02)], p=0.7),
    T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.2))], p=0.2),
    T.RandomAffine(degrees=2, translate=(0.02, 0.05), scale=(0.95, 1.05), shear=1, fill=0),
    T.ToTensor(),
    T.RandomErasing(p=0.3, scale=(0.02, 0.15)),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

val_tfm = T.Compose([
    T.Resize((cfg.img_h, cfg.img_w)),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

class ProvinceDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tfm):
        self.df = df
        self.tfm = tfm

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        x = self.tfm(img)
        y = int(row['y'])
        return x, y

tr_ds = ProvinceDataset(tr_df, train_tfm)
va_ds = ProvinceDataset(va_df, val_tfm)

tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, persistent_workers=cfg.persistent_workers, pin_memory=True, drop_last=True)
va_loader = DataLoader(va_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, persistent_workers=cfg.persistent_workers, pin_memory=True)

print('train batches:', len(tr_loader), 'val batches:', len(va_loader))

train batches: 548 val batches: 61


In [35]:
# Cell 9: Model + optimizer + scheduler
model = timm.create_model(cfg.model_name, pretrained=True, num_classes=cfg.num_classes, in_chans=3, drop_rate=0.3).to(device)

counts = tr_df['y'].value_counts().sort_index().values.astype(np.float32)
weights = (counts.sum() / np.maximum(counts, 1.0))
weights = weights / weights.mean()
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

steps_per_epoch = len(tr_loader)
total_steps = steps_per_epoch * cfg.epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(total_steps, 1))

scaler = torch.cuda.amp.GradScaler(enabled=(cfg.amp and device == 'cuda'))

def acc_top1(logits: torch.Tensor, y: torch.Tensor) -> float:
    pred = logits.argmax(dim=1)
    return (pred == y).float().mean().item()

print('model:', cfg.model_name, 'num_classes:', cfg.num_classes)

model: mobilenetv3_small_100 num_classes: 78


/tmp/ipython-input-506784733.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(cfg.amp and device == 'cuda'))


In [36]:
# Cell 10: Train loop (saves best checkpoint)
def run_one_epoch(model, loader, train: bool, max_batches: int | None = None):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_acc = 0.0
    n = 0

    pbar = tqdm(loader, leave=False)
    for step, (x, y) in enumerate(pbar, start=1):
        if max_batches is not None and step > max_batches:
            break

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(cfg.amp and device == 'cuda')):
            logits = model(x)
            loss = criterion(logits, y)

        if train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_acc += acc_top1(logits.detach(), y) * bs
        n += bs

        lr = optimizer.param_groups[0]["lr"]
        pbar.set_postfix_str(f"loss={total_loss/max(n,1):.4f} acc={total_acc/max(n,1):.4f} lr={lr:.2e}")

    return total_loss / max(n, 1), total_acc / max(n, 1)

best_val_acc = -1.0
history = []
best_path = os.path.join(cfg.save_dir, 'best.pt')
patience = 3
no_improve = 0

for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_one_epoch(model, tr_loader, train=True, max_batches=(cfg.max_train_batches if cfg.debug_run else None))
    va_loss, va_acc = run_one_epoch(model, va_loader, train=False, max_batches=(cfg.max_val_batches if cfg.debug_run else None))

    row = {'epoch': epoch, 'train_loss': tr_loss, 'train_acc': tr_acc, 'val_loss': va_loss, 'val_acc': va_acc, 'secs': time.time() - t0}
    history.append(row)
    print(row)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        ckpt = {'model_name': cfg.model_name, 'num_classes': cfg.num_classes, 'state_dict': model.state_dict(), 'label2idx': label2idx, 'idx2label': idx2label, 'epoch': epoch, 'val_acc': float(va_acc)}
        torch.save(ckpt, best_path)
        print(f'Saved best -> {best_path} (val_acc={va_acc:.4f})')
        no_improve = 0
    else :
        no_improve += 1
        if no_improve >= patience:
            print(f'Early stopping at epoch {epoch} due to no improvement in val_acc for {patience} consecutive epochs.')
            break

pd.DataFrame(history).to_csv(os.path.join(cfg.save_dir, 'history.csv'), index=False)
print('best_val_acc:', best_val_acc)

  0%|          | 0/548 [00:00<?, ?it/s]

/tmp/ipython-input-1378144797.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(cfg.amp and device == 'cuda')):
/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


  0%|          | 0/61 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 1.9673034269770566, 'train_acc': 0.5362967609489051, 'val_loss': 0.044680494551475235, 'val_acc': 0.9870512832128084, 'secs': 259.6216161251068}
Saved best -> /content/drive/MyDrive/ALPR_Project/alpr_province_classifier_final/best.pt (val_acc=0.9871)


  0%|          | 0/548 [00:00<?, ?it/s]

...

  0%|          | 0/61 [00:00<?, ?it/s]

{'epoch': 18, 'train_loss': 0.01965431980097226, 'train_acc': 0.9932781135948905, 'val_loss': 0.0017127115805077408, 'val_acc': 0.999423076923077, 'secs': 216.4557604789734}


  0%|          | 0/548 [00:00<?, ?it/s]

  0%|          | 0/61 [00:00<?, ?it/s]

{'epoch': 19, 'train_loss': 0.018298781959995982, 'train_acc': 0.9937485743613139, 'val_loss': 0.001756092662830596, 'val_acc': 0.999423076923077, 'secs': 214.21500182151794}
Early stopping at epoch 19 due to no improvement in val_acc for 3 consecutive epochs.
best_val_acc: 0.9995512820512821


In [37]:
# Cell 11: Evaluate best checkpoint on lower_test + lower_test_synthetic
@torch.no_grad()
def eval_dataset(dataset_dir: str, name: str):
    df = read_labels(dataset_dir)
    df['y'] = df['label'].map(label2idx)
    df = df[df['y'].notna()].copy()
    df['y'] = df['y'].astype(int)

    if cfg.debug_run and len(df) > cfg.debug_eval_rows:
        df = df.sample(cfg.debug_eval_rows, random_state=42).reset_index(drop=True)
        print(f'[debug] {name}: sampled to {len(df)} rows')

    ds = ProvinceDataset(df, val_tfm)
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=(device=='cuda'))

    model.eval()
    total = 0
    correct = 0
    for x, y in tqdm(loader, desc=f'eval:{name}'):
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.numel()

    acc = correct / max(total, 1)
    print(f'[{name}] n={total} acc={acc:.4f}')
    return acc

best_path = os.path.join(cfg.save_dir, 'best.pt')
ckpt = torch.load(best_path, map_location=device)
model = timm.create_model(ckpt['model_name'], pretrained=False, num_classes=ckpt['num_classes'], in_chans=3).to(device)
model.load_state_dict(ckpt['state_dict'])
model.eval()

_ = eval_dataset(TEST_DIR, 'lower_test')
_ = eval_dataset(TEST_SYN_DIR, 'lower_test_synthetic')

[lower_test] removed excluded_labels: 1


eval:lower_test:   0%|          | 0/28 [00:00<?, ?it/s]

[lower_test] n=7077 acc=0.9936


eval:lower_test_synthetic:   0%|          | 0/61 [00:00<?, ?it/s]

[lower_test_synthetic] n=15600 acc=0.9923


In [38]:
# Cell 12: Inference helper (เอาไปใช้กับ RTSP pipeline ได้)
@torch.no_grad()
def predict_province(image_path: str, topk: int = 5):
    img = Image.open(image_path).convert('RGB')
    x = val_tfm(img).unsqueeze(0).to(device)
    logits = model(x)
    probs = F.softmax(logits, dim=1).squeeze(0)
    vals, idxs = torch.topk(probs, k=min(topk, probs.numel()))
    return [(idx2label[int(i)], float(v)) for v, i in zip(vals.cpu(), idxs.cpu())]